<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/Mini_Project_MCP_Agents_Gemini_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini-Project — MCP + Agents AI avec Gemini

## Dev Workspace Assistant

Ce notebook construit une application agentique de bout en bout dans **Google Colab**.

L’agent Gemini peut :

- explorer un projet avec un serveur MCP **Filesystem** ;
- analyser son dépôt avec un serveur MCP **Git** ;
- appeler un serveur MCP Python personnalisé pour calculer des métriques et contrôler la qualité ;
- décider lui-même de l’ordre des outils à utiliser ;
- produire un rapport d’audit dans le dossier du projet ;
- comparer ses résultats à une baseline sans outils.

> Le notebook ne contient aucun nom d’étudiant ni aucune clé API.

## 1. Architecture

```mermaid
flowchart LR
    U[Utilisateur] --> A[Agent Gemini]
    A --> F[Serveur MCP Filesystem]
    A --> G[Serveur MCP Git]
    A --> C[Serveur MCP personnalisé]
    F --> W[(Projet de démonstration)]
    G --> W
    C --> W
    A --> R[Rapport AGENT_REPORT.md]
```

### Responsabilités des serveurs

| Serveur | Rôle |
|---|---|
| Filesystem MCP | Lire, rechercher et écrire les fichiers autorisés |
| Git MCP | Lire le statut, l’historique et les différences Git |
| Custom MCP | Calculer les métriques du projet et contrôler un rapport Markdown |
| Gemini | Choisir les outils, interpréter les résultats et construire la réponse finale |

La logique métier n’impose pas une séquence fixe d’appels. Le modèle reçoit un objectif, la liste des outils disponibles et une politique de sécurité.

## 2. Installation des dépendances

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "nest_asyncio" \
  "fastmcp>=2.0.0" \
  "mcp-server-git>=2025.12.18"

Après l’installation, redémarrez l’environnement d’exécution uniquement si Colab le demande.

La version récente de `mcp-server-git` est demandée afin d’éviter d’utiliser d’anciennes versions ayant reçu des correctifs de sécurité.

In [ ]:
import asyncio
import json
import os
import subprocess
import sys
import textwrap
from getpass import getpass
from pathlib import Path

import nest_asyncio

nest_asyncio.apply()

print("Python :", sys.version.split()[0])

## 3. Configuration sécurisée de la clé Gemini

In [ ]:
# Méthode recommandée :
# 1. Dans Colab, ouvrez le panneau « Secrets ».
# 2. Ajoutez un secret nommé GOOGLE_API_KEY.
# 3. Activez l'accès du notebook à ce secret.

try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    api_key = None

if not api_key:
    api_key = getpass("Entrez votre GOOGLE_API_KEY : ").strip()

if not api_key:
    raise ValueError("GOOGLE_API_KEY est obligatoire pour exécuter Gemini.")

os.environ["GOOGLE_API_KEY"] = api_key
os.environ.setdefault("GEMINI_MODEL", "gemini-2.5-flash")

print("Clé configurée :", bool(os.environ.get("GOOGLE_API_KEY")))
print("Modèle :", os.environ["GEMINI_MODEL"])

## 4. Vérification de Node, NPM et Git

In [ ]:
!node --version
!npx --version
!git --version

Si Node ou NPM est absent, exécutez la cellule suivante. Elle est généralement inutile dans Colab.

In [ ]:
# À décommenter uniquement en cas d'erreur dans la cellule précédente.
# !apt-get -qq update
# !apt-get -qq install -y nodejs npm git
# !node --version
# !npx --version
# !git --version

## 5. Création d’un projet Git de démonstration

In [ ]:
WORKDIR = Path("/content/mcp_dev_workspace").resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)

readme = """# Task Manager

Petit projet Python permettant d'ajouter et de terminer des tâches.

## Fonctionnalités

- Ajouter une tâche
- Terminer une tâche
- Exécuter les tests avec pytest

## Limites actuelles

Les tâches sont uniquement conservées en mémoire.
"""

task_manager = """\"\"\"Gestionnaire de tâches minimal.\"\"\"

from typing import Any


def add_task(tasks: list[dict[str, Any]], title: str) -> dict[str, Any]:
    \"\"\"Ajoute une tâche et retourne la tâche créée.\"\"\"
    clean_title = title.strip()
    if not clean_title:
        raise ValueError("Le titre ne peut pas être vide.")

    task = {
        "id": len(tasks) + 1,
        "title": clean_title,
        "done": False,
    }
    tasks.append(task)
    return task


def complete_task(tasks: list[dict[str, Any]], task_id: int) -> dict[str, Any]:
    \"\"\"Marque une tâche comme terminée.\"\"\"
    for task in tasks:
        if task["id"] == task_id:
            task["done"] = True
            return task

    raise KeyError(f"Tâche introuvable : {task_id}")
"""

tests = """from task_manager import add_task, complete_task


def test_add_task():
    tasks = []
    task = add_task(tasks, "Préparer le rapport")
    assert task["id"] == 1
    assert task["title"] == "Préparer le rapport"
    assert task["done"] is False


def test_complete_task():
    tasks = []
    add_task(tasks, "Tester l'application")
    completed = complete_task(tasks, 1)
    assert completed["done"] is True
"""

(WORKDIR / "README.md").write_text(readme, encoding="utf-8")
(WORKDIR / "task_manager.py").write_text(task_manager, encoding="utf-8")
(WORKDIR / "test_task_manager.py").write_text(tests, encoding="utf-8")
(WORKDIR / ".gitignore").write_text("__pycache__/\n.pytest_cache/\n", encoding="utf-8")

def run_command(command: list[str], cwd: Path = WORKDIR) -> str:
    result = subprocess.run(
        command,
        cwd=cwd,
        capture_output=True,
        text=True,
        check=True,
    )
    return result.stdout.strip()

if not (WORKDIR / ".git").exists():
    run_command(["git", "init"])
    run_command(["git", "config", "user.email", "student@example.local"])
    run_command(["git", "config", "user.name", "Colab Student"])
    run_command(["git", "add", "."])
    run_command(["git", "commit", "-m", "Initial task manager implementation"])

history = run_command(["git", "log", "--oneline"])
if "Add project usage notes" not in history:
    with (WORKDIR / "README.md").open("a", encoding="utf-8") as file:
        file.write("\n## Utilisation\n\nImporter les fonctions depuis `task_manager.py`.\n")
    run_command(["git", "add", "README.md"])
    run_command(["git", "commit", "-m", "Add project usage notes"])

current_source = (WORKDIR / "task_manager.py").read_text(encoding="utf-8")
todo_line = "\n\n# TODO: ajouter une persistance JSON et des tests pour les erreurs.\n"
if "TODO: ajouter une persistance JSON" not in current_source:
    with (WORKDIR / "task_manager.py").open("a", encoding="utf-8") as file:
        file.write(todo_line)

print("Projet créé dans :", WORKDIR)
print("\nFichiers :")
for path in sorted(WORKDIR.iterdir()):
    if path.name != ".git":
        print("-", path.name)

print("\nHistorique Git :")
print(run_command(["git", "log", "--oneline", "--max-count=5"]))

print("\nStatut Git :")
print(run_command(["git", "status", "--short"]) or "Aucune modification")

## 6. Tests locaux du projet de démonstration

In [ ]:
!cd "$WORKDIR" && python -m pytest -q

## 7. Création du serveur MCP personnalisé

In [ ]:
CUSTOM_SERVER_PATH = Path("/content/custom_mcp_server.py")

custom_server_source = 'from collections import Counter\nfrom pathlib import Path\nfrom typing import Any\n\nfrom fastmcp import FastMCP\n\nmcp = FastMCP(name="custom_ops")\n\nALLOWED_ROOT = Path("/content/mcp_dev_workspace").resolve()\n\n\ndef secure_project_path(project_path: str) -> Path:\n    """Retourne un chemin autorisé et bloque les sorties du dossier racine."""\n    path = Path(project_path).resolve()\n    try:\n        path.relative_to(ALLOWED_ROOT)\n    except ValueError as exc:\n        raise ValueError(\n            f"Chemin interdit. Le projet doit rester dans {ALLOWED_ROOT}"\n        ) from exc\n\n    if not path.exists() or not path.is_dir():\n        raise ValueError(f"Dossier de projet introuvable : {path}")\n    return path\n\n\n@mcp.tool\ndef ping() -> str:\n    """Vérifie que le serveur personnalisé fonctionne."""\n    return "pong"\n\n\n@mcp.tool\ndef project_metrics(project_path: str) -> dict[str, Any]:\n    """Calcule des métriques simples sur un projet texte ou Python."""\n    root = secure_project_path(project_path)\n\n    ignored_parts = {".git", "__pycache__", ".pytest_cache"}\n    extension_counts: Counter[str] = Counter()\n    total_lines = 0\n    todo_count = 0\n    test_files = 0\n    readable_files = 0\n\n    for path in root.rglob("*"):\n        if not path.is_file() or any(part in ignored_parts for part in path.parts):\n            continue\n\n        extension_counts[path.suffix or "[no_extension]"] += 1\n        if path.name.startswith("test_") or path.name.endswith("_test.py"):\n            test_files += 1\n\n        try:\n            text = path.read_text(encoding="utf-8")\n        except (UnicodeDecodeError, OSError):\n            continue\n\n        readable_files += 1\n        total_lines += len(text.splitlines())\n        todo_count += text.upper().count("TODO")\n\n    return {\n        "project_path": str(root),\n        "readable_files": readable_files,\n        "total_lines": total_lines,\n        "test_files": test_files,\n        "todo_mentions": todo_count,\n        "extensions": dict(sorted(extension_counts.items())),\n    }\n\n\n@mcp.tool\ndef build_changelog(commit_messages: list[str]) -> str:\n    """Transforme des messages de commits en changelog Markdown."""\n    cleaned = [message.strip() for message in commit_messages if message.strip()]\n    if not cleaned:\n        return "## Changelog\\n\\n_Aucun commit fourni._"\n\n    lines = ["## Changelog", ""]\n    lines.extend(f"- {message}" for message in cleaned)\n    return "\\n".join(lines)\n\n\n@mcp.tool\ndef quality_gate(report_markdown: str) -> dict[str, Any]:\n    """Contrôle la présence des sections attendues dans un rapport Markdown."""\n    required_sections = [\n        "Résumé exécutif",\n        "Fichiers analysés",\n        "Analyse Git",\n        "Qualité et tests",\n        "Risques",\n        "Recommandations",\n        "Changelog",\n    ]\n\n    missing = [\n        section\n        for section in required_sections\n        if section.lower() not in report_markdown.lower()\n    ]\n\n    return {\n        "passed": len(missing) == 0 and len(report_markdown) >= 700,\n        "missing_sections": missing,\n        "character_count": len(report_markdown),\n        "has_markdown_title": report_markdown.lstrip().startswith("#"),\n    }\n\n\nif __name__ == "__main__":\n    mcp.run(transport="stdio")\n'

CUSTOM_SERVER_PATH.write_text(
    custom_server_source,
    encoding="utf-8",
)

print("Serveur écrit dans :", CUSTOM_SERVER_PATH)
print("Taille :", CUSTOM_SERVER_PATH.stat().st_size, "octets")

### Outils personnalisés exposés

- `ping()` : contrôle de santé ;
- `project_metrics(project_path)` : métriques sur les fichiers, lignes, tests et TODO ;
- `build_changelog(commit_messages)` : génération d’un changelog Markdown ;
- `quality_gate(report_markdown)` : validation de la structure du rapport.

## 8. Connexion aux trois serveurs MCP

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": "npx",
        "args": [
            "-y",
            "@modelcontextprotocol/server-filesystem",
            str(WORKDIR),
        ],
    },
    "git": {
        "transport": "stdio",
        "command": "python",
        "args": [
            "-m",
            "mcp_server_git",
            "--repository",
            str(WORKDIR),
        ],
    },
    "custom_ops": {
        "transport": "stdio",
        "command": "python",
        "args": [str(CUSTOM_SERVER_PATH)],
    },
}

client = MultiServerMCPClient(
    mcp_connections,
    tool_name_prefix=True,
)

tools = await client.get_tools()

print("Nombre total d'outils :", len(tools))
for tool in sorted(tools, key=lambda item: item.name):
    print(f"- {tool.name}: {tool.description[:100]}")

## 9. Contrôles d’intégration MCP

In [ ]:
tool_names = {tool.name for tool in tools}

server_checks = {
    "filesystem": any(name.startswith("filesystem_") for name in tool_names),
    "git": any(name.startswith("git_") for name in tool_names),
    "custom_ops": any(name.startswith("custom_ops_") for name in tool_names),
}

print(json.dumps(server_checks, indent=2, ensure_ascii=False))
assert all(server_checks.values()), "Au moins un serveur MCP n'a pas chargé ses outils."

ping_tool = next(
    tool for tool in tools
    if tool.name == "custom_ops_ping"
)
print("Réponse du serveur personnalisé :", await ping_tool.ainvoke({}))

## 10. Baseline : Gemini sans outils

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model=os.environ["GEMINI_MODEL"],
    temperature=0,
)

baseline_prompt = """
Tu dois auditer un petit projet Python nommé Task Manager.
Sans utiliser d'outil et sans voir les fichiers, donne :
1. les fichiers probablement présents ;
2. l'état Git ;
3. les défauts exacts ;
4. un changelog basé sur les vrais commits.
Indique clairement ce que tu ne peux pas vérifier.
"""

baseline_response = await llm.ainvoke(baseline_prompt)
print(baseline_response.content)

La baseline ne dispose d’aucune preuve issue du projet. Elle doit normalement reconnaître qu’elle ne peut pas connaître précisément les fichiers, le statut Git ou les commits.

## 11. Construction de l’agent Gemini outillé

In [ ]:
SYSTEM_PROMPT = f"""
Tu es un Dev Workspace Assistant responsable d'auditer un projet local.

Dossier autorisé : {WORKDIR}
Rapport final obligatoire : {WORKDIR / "AGENT_REPORT.md"}

Règles :
- Utilise les outils MCP et ne prétends jamais avoir lu une information sans preuve.
- Utilise au moins un outil Filesystem, un outil Git et un outil custom_ops.
- Tu décides toi-même de l'ordre et du nombre d'appels.
- Inspecte les fichiers importants, le statut Git, le diff non commité et les commits récents.
- Utilise custom_ops_project_metrics.
- Utilise custom_ops_build_changelog à partir des vrais messages de commits.
- Prépare un rapport Markdown d'au moins 700 caractères.
- Le rapport doit contenir exactement les sections suivantes :
  Résumé exécutif, Fichiers analysés, Analyse Git, Qualité et tests,
  Risques, Recommandations, Changelog.
- Appelle custom_ops_quality_gate avant d'enregistrer la version finale.
- Si le contrôle échoue, corrige le rapport puis contrôle-le à nouveau.
- Enregistre le rapport final avec le serveur Filesystem.
- Ne modifie aucun autre fichier du projet.
- N'effectue aucun commit.
"""

try:
    from langchain.agents import create_agent

    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt=SYSTEM_PROMPT,
    )
    AGENT_API = "langchain.create_agent"
except ImportError:
    from langgraph.prebuilt import create_react_agent

    agent = create_react_agent(
        model=llm,
        tools=tools,
        prompt=SYSTEM_PROMPT,
    )
    AGENT_API = "langgraph.create_react_agent"

print("API agent :", AGENT_API)

## 12. Exécution du scénario agentique

In [ ]:
USER_REQUEST = f"""
Audite complètement le projet situé dans {WORKDIR}.
Crée le fichier AGENT_REPORT.md demandé par la politique système.
Dans ta réponse finale, résume les trois problèmes les plus importants
et confirme le chemin du rapport.
"""

result = await agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": USER_REQUEST,
            }
        ]
    },
    config={"recursion_limit": 40},
)

final_message = result["messages"][-1]
final_content = final_message.content

if isinstance(final_content, str):
    print(final_content)
else:
    print(json.dumps(final_content, indent=2, ensure_ascii=False, default=str))

## 13. Trace des décisions et appels d’outils

In [ ]:
called_tools = []

for index, message in enumerate(result["messages"], start=1):
    tool_calls = getattr(message, "tool_calls", None) or []
    for call in tool_calls:
        name = call.get("name", "outil_inconnu")
        called_tools.append(name)
        print(f"{index:02d}. Appel : {name}")
        print(
            "    Arguments :",
            json.dumps(
                call.get("args", {}),
                ensure_ascii=False,
                default=str,
            )[:500],
        )

print("\nOutils appelés :", len(called_tools))
print(json.dumps(called_tools, indent=2, ensure_ascii=False))

## 14. Vérification du rapport produit

In [ ]:
REPORT_PATH = WORKDIR / "AGENT_REPORT.md"

if not REPORT_PATH.exists():
    raise FileNotFoundError(
        "Le rapport n'a pas été créé. Relancez la cellule de l'agent "
        "avec un modèle Gemini compatible avec les outils."
    )

report_text = REPORT_PATH.read_text(encoding="utf-8")

print("Rapport :", REPORT_PATH)
print("Taille :", len(report_text), "caractères")
print("\n" + "=" * 80 + "\n")
print(report_text)

## 15. Évaluation automatique

In [ ]:
required_sections = [
    "Résumé exécutif",
    "Fichiers analysés",
    "Analyse Git",
    "Qualité et tests",
    "Risques",
    "Recommandations",
    "Changelog",
]

evaluation = {
    "rapport_cree": REPORT_PATH.exists(),
    "longueur_minimale": len(report_text) >= 700,
    "sections_completes": all(
        section.lower() in report_text.lower()
        for section in required_sections
    ),
    "filesystem_utilise": any(
        name.startswith("filesystem_")
        for name in called_tools
    ),
    "git_utilise": any(
        name.startswith("git_")
        for name in called_tools
    ),
    "custom_mcp_utilise": any(
        name.startswith("custom_ops_")
        for name in called_tools
    ),
    "mentionne_un_commit_reel": (
        "Initial task manager implementation" in report_text
        or "Add project usage notes" in report_text
    ),
    "mentionne_le_todo_reel": "TODO" in report_text.upper(),
}

score = sum(evaluation.values())
total = len(evaluation)

for criterion, passed in evaluation.items():
    icon = "✅" if passed else "❌"
    print(f"{icon} {criterion}")

print(f"\nScore automatique : {score}/{total} ({score / total:.0%})")

## 16. Comparaison quantitative baseline vs agent MCP

In [ ]:
baseline_text = (
    baseline_response.content
    if isinstance(baseline_response.content, str)
    else str(baseline_response.content)
)

comparison = {
    "baseline_sans_outils": {
        "acces_aux_fichiers_reels": False,
        "acces_a_git": False,
        "rapport_enregistre": False,
        "preuves_verifiables": 0,
    },
    "agent_mcp": {
        "acces_aux_fichiers_reels": evaluation["filesystem_utilise"],
        "acces_a_git": evaluation["git_utilise"],
        "rapport_enregistre": evaluation["rapport_cree"],
        "preuves_verifiables": sum([
            evaluation["mentionne_un_commit_reel"],
            evaluation["mentionne_le_todo_reel"],
            evaluation["sections_completes"],
        ]),
    },
}

print(json.dumps(comparison, indent=2, ensure_ascii=False))

## 17. Analyse des résultats

### Pourquoi l’agent MCP est meilleur que la baseline

La baseline peut produire des recommandations générales, mais elle ne peut pas vérifier l’état réel du projet. L’agent MCP peut, lui :

1. lire les fichiers réellement présents ;
2. consulter les commits et les modifications non enregistrées ;
3. calculer des métriques déterministes ;
4. contrôler la structure du rapport ;
5. enregistrer un livrable dans le workspace.

### Limites

- la qualité du raisonnement dépend du modèle Gemini choisi ;
- les appels de fonctions peuvent varier légèrement entre deux exécutions ;
- le projet de démonstration est volontairement petit ;
- les serveurs locaux doivent rester limités à un dossier autorisé ;
- un agent ne doit pas recevoir de droits d’écriture plus larges que nécessaire.

### Garde-fous appliqués

- accès Filesystem limité à `WORKDIR` ;
- serveur Git limité au dépôt de démonstration ;
- validation du chemin dans le serveur personnalisé ;
- interdiction de commit dans la politique système ;
- clé API saisie dans les secrets Colab ou avec `getpass` ;
- évaluation automatique des serveurs réellement utilisés.

## 18. Démonstration libre

In [ ]:
custom_request = f"""
Analyse le projet {WORKDIR}.
Explique les changements non commités, propose trois améliorations prioritaires
et indique quels fichiers devraient recevoir de nouveaux tests.
Ne modifie aucun fichier.
"""

custom_result = await agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": custom_request,
            }
        ]
    },
    config={"recursion_limit": 30},
)

custom_final = custom_result["messages"][-1].content
print(
    custom_final
    if isinstance(custom_final, str)
    else json.dumps(
        custom_final,
        indent=2,
        ensure_ascii=False,
        default=str,
    )
)

## 19. Checklist de remise

- [x] Notebook Google Colab
- [x] Gemini configuré avec `GOOGLE_API_KEY`
- [x] Deux serveurs MCP tiers : Filesystem et Git
- [x] Un serveur MCP personnalisé en Python
- [x] Connexion avec `MultiServerMCPClient`
- [x] Agent choisissant dynamiquement ses outils
- [x] Composition entre trois serveurs
- [x] Rapport enregistré dans le workspace
- [x] Baseline sans outils
- [x] Comparaison quantitative
- [x] Tests et contrôles d’intégration
- [x] Limites et garde-fous documentés

### Fichiers créés pendant l’exécution

- `/content/mcp_dev_workspace/` : projet Git de démonstration ;
- `/content/custom_mcp_server.py` : serveur MCP personnalisé ;
- `/content/mcp_dev_workspace/AGENT_REPORT.md` : rapport produit par l’agent.